# Flickr30k T3→S3 Conditional Deployment Lock

Khóa package `encoder FP32 + decoder dynamic INT8` cho benchmark Android.
Đây là khóa **có điều kiện**: chất lượng Validation và kiểm định không-kém-hơn
đạt, nhưng caption trùng nguyên văn chỉ 75,64% và kết luận không-kém-hơn rất
sát biên. Notebook không cấp phép phát hành production vô điều kiện và không
đọc Held-out Test.

## Kaggle input bắt buộc — đúng 4 input

1. Output `F30K_T3S3_INT8_Preflight`.
2. Output `F30K_T3S3_INT8_Validation`.
3. Output `F30K_T3S3_INT8_NonInferiority`.
4. Output `F30K_Data`.

Runtime: CPU / Accelerator None / Internet Off, dự kiến dưới 2 phút.

## Cell 1 — Chain authorization and artifact integrity

In [1]:
import hashlib
import json
import os
import shutil
from datetime import datetime, timezone
from pathlib import Path

EXPERIMENT_ID = "F30K_T3S3_Deployment_Lock"
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") / EXPERIMENT_ID
PACKAGE_ROOT = WORK_ROOT / "mobile_package"
PACKAGE_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json_atomic(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def unique_named(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (f"Cần đúng một {name}", [str(x) for x in matches])
    return matches[0]

PREFLIGHT_RESULT_PATH = unique_named(
    "F30K_T3S3_INT8_Preflight_RESULT.json"
)
VALIDATION_RESULT_PATH = unique_named(
    "F30K_T3S3_INT8_Validation_RESULT.json"
)
NONINFERIORITY_RESULT_PATH = unique_named(
    "F30K_T3S3_INT8_NonInferiority_RESULT.json"
)
DATA_CONTRACT_PATH = unique_named("F30K_DATA_CONTRACT.json")
VOCABULARY_PATH = unique_named("f30k_vocabulary_minfreq5.json")

PREFLIGHT = read_json(PREFLIGHT_RESULT_PATH)
VALIDATION = read_json(VALIDATION_RESULT_PATH)
NONINFERIORITY = read_json(NONINFERIORITY_RESULT_PATH)
DATA_CONTRACT = read_json(DATA_CONTRACT_PATH)
VOCABULARY = read_json(VOCABULARY_PATH)

assert PREFLIGHT["status"] == "passed"
assert PREFLIGHT["selected_candidate"] == (
    "mixed_fp32_encoder_dynamic_int8_decoder"
)
assert VALIDATION["status"] == "passed_with_caption_variation_review_required"
assert VALIDATION["gate_interpretation"]["strict_original_gate_passed"] is False
assert VALIDATION["gate_interpretation"]["scientific_quality_gate_passed"] is True
assert NONINFERIORITY["status"] == "passed"
assert NONINFERIORITY["bootstrap"]["noninferiority_passed"] is True
assert NONINFERIORITY["bootstrap"]["superiority_exploratory"] is False
assert NONINFERIORITY["original_strict_gate"]["passed"] is False
assert NONINFERIORITY["original_strict_gate"][
    "preserved_without_threshold_change"
] is True
assert NONINFERIORITY["next_phase_authorized"] == EXPERIMENT_ID
assert DATA_CONTRACT["test_set_status"] == "sealed"
assert DATA_CONTRACT["test_caption_exported"] is False
assert VOCABULARY["special_token_ids"] == {
    "pad": 0, "bos": 1, "eos": 2, "unk": 3
}
assert len(VOCABULARY["tokens"]) == 7456
assert sha256(VOCABULARY_PATH) == DATA_CONTRACT["artifacts"]["vocabulary"]["sha256"]

assert NONINFERIORITY["source_result_sha256"] == sha256(VALIDATION_RESULT_PATH)
assert VALIDATION["source_integrity"]["preflight_result_sha256"] == sha256(
    PREFLIGHT_RESULT_PATH
)

ENCODER_SOURCE = (
    PREFLIGHT_RESULT_PATH.parent / "f30k_t3s3_seed42_encoder_fp32.onnx"
)
DECODER_SOURCE = (
    PREFLIGHT_RESULT_PATH.parent
    / "f30k_t3s3_seed42_decoder_dynamic_int8.onnx"
)
for path in [ENCODER_SOURCE, DECODER_SOURCE]:
    record = PREFLIGHT["artifacts"][path.name]
    assert path.is_file()
    assert path.stat().st_size == record["bytes"]
    assert sha256(path) == record["sha256"]

print("Authorization          : PASSED")
print("Scientific quality     : PASSED")
print("Non-inferiority        : PASSED")
print("Superiority            : NOT ESTABLISHED")
print("Strict verbatim gate   : FAILED / PRESERVED")
print("Held-out Test          : SEALED / NOT READ")

Authorization          : PASSED
Scientific quality     : PASSED
Non-inferiority        : PASSED
Superiority            : NOT ESTABLISHED
Strict verbatim gate   : FAILED / PRESERVED
Held-out Test          : SEALED / NOT READ


## Cell 2 — Immutable mobile package

In [2]:
ENCODER_TARGET = PACKAGE_ROOT / ENCODER_SOURCE.name
DECODER_TARGET = PACKAGE_ROOT / DECODER_SOURCE.name
VOCABULARY_TARGET = PACKAGE_ROOT / VOCABULARY_PATH.name
for source, target in [
    (ENCODER_SOURCE, ENCODER_TARGET),
    (DECODER_SOURCE, DECODER_TARGET),
    (VOCABULARY_PATH, VOCABULARY_TARGET),
]:
    shutil.copy2(source, target)
    assert sha256(source) == sha256(target)

deployment_config = {
    "schema_version": "f30k_t3s3_mobile_config_v1",
    "model": {
        "pair": "T3 SigLIP ViT-B/16 adapted -> S3 TinyViT-5M",
        "method": "Direct Logit-KD",
        "seed": 42,
        "encoder": {
            "file": ENCODER_TARGET.name,
            "precision": "FP32",
            "input_name": "images",
            "input_shape": [1, 3, 224, 224],
            "output_name": "memory",
            "output_shape": [1, 49, 256],
        },
        "decoder": {
            "file": DECODER_TARGET.name,
            "precision": "dynamic per-channel QInt8 weights",
            "token_ids_shape": [1, 31],
            "memory_shape": [1, 49, 256],
            "attention_mask_shape": [1, 31],
            "logits_shape": [1, 31, 7456],
        },
    },
    "preprocessing": {
        "color_space": "RGB",
        "resize_shorter_side": 256,
        "resize_interpolation": "bicubic",
        "center_crop": [224, 224],
        "scale": "uint8 / 255.0",
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225],
        "tensor_layout": "NCHW",
    },
    "decoding": {
        "beam_size": 3,
        "length_penalty": 0.7,
        "maximum_generated_tokens": 30,
        "fixed_token_buffer_length": 31,
        "pad_id": 0,
        "bos_id": 1,
        "eos_id": 2,
        "unk_id": 3,
    },
    "vocabulary": {
        "file": VOCABULARY_TARGET.name,
        "size": 7456,
        "sha256": sha256(VOCABULARY_TARGET),
    },
}
CONFIG_PATH = PACKAGE_ROOT / "F30K_T3S3_MOBILE_CONFIG.json"
write_json_atomic(CONFIG_PATH, deployment_config)

package_bytes = (
    ENCODER_TARGET.stat().st_size
    + DECODER_TARGET.stat().st_size
    + VOCABULARY_TARGET.stat().st_size
    + CONFIG_PATH.stat().st_size
)
print("Package files          : 4")
print("Package bytes          :", package_bytes)
print("Package MiB            :", package_bytes / 1024**2)
print("Model-only MiB         :", PREFLIGHT["selected_size"]["selected_MiB"])

Package files          : 4
Package bytes          : 28765524
Package MiB            : 27.432941436767578
Model-only MiB         : 27.182700157165527


## Cell 3 — Conditional lock and parallel next phases

In [3]:
ci_lower = float(NONINFERIORITY["bootstrap"]["ci95_lower"])
margin = float(NONINFERIORITY["bootstrap"]["noninferiority_margin"])
boundary_clearance = ci_lower - margin
assert boundary_clearance > 0

package_artifacts = []
for path in [
    ENCODER_TARGET,
    DECODER_TARGET,
    VOCABULARY_TARGET,
    CONFIG_PATH,
]:
    package_artifacts.append({
        "relative_path": f"mobile_package/{path.name}",
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    })

lock = {
    "schema_version": "f30k_t3s3_conditional_deployment_lock_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "locked_for_conditional_mobile_benchmark",
    "experiment_id": EXPERIMENT_ID,
    "pair": VALIDATION["pair"],
    "method": VALIDATION["method"],
    "seed": VALIDATION["seed"],
    "selected_precision": PREFLIGHT["selected_precision"],
    "validation_metrics": VALIDATION["metrics"],
    "size": PREFLIGHT["selected_size"],
    "noninferiority": NONINFERIORITY["bootstrap"],
    "decision": {
        "scientific_quality_gate": "passed",
        "paired_CIDEr_D_noninferiority": "passed",
        "superiority": "not established",
        "strict_verbatim_stability": "failed",
        "mobile_benchmark": "authorized",
        "unconditional_production_release": "not authorized",
    },
    "boundary_risk": {
        "ci95_lower_minus_noninferiority_margin": boundary_clearance,
        "interpretation": (
            "The non-inferiority conclusion passes by a very small margin and "
            "must be reported as boundary-sensitive."
        ),
    },
    "source_integrity": {
        "preflight_result_sha256": sha256(PREFLIGHT_RESULT_PATH),
        "validation_result_sha256": sha256(VALIDATION_RESULT_PATH),
        "noninferiority_result_sha256": sha256(NONINFERIORITY_RESULT_PATH),
        "data_contract_sha256": sha256(DATA_CONTRACT_PATH),
    },
    "package_artifacts": package_artifacts,
    "method_or_threshold_changes_after_result": False,
    "held_out_test_authorized": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "parallel_next_phases": [
        "F30K_T3S3_Android_Benchmark",
        "F30K_T3S3_INT8_Qualitative",
    ],
}
LOCK_PATH = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
write_json_atomic(LOCK_PATH, lock)

manifest_files = [
    ENCODER_TARGET,
    DECODER_TARGET,
    VOCABULARY_TARGET,
    CONFIG_PATH,
    LOCK_PATH,
]
write_json_atomic(WORK_ROOT / "MANIFEST.json", {
    "schema_version": "f30k_t3s3_deployment_lock_manifest_v1",
    "experiment_id": EXPERIMENT_ID,
    "status": "complete",
    "artifacts": [
        {
            "relative_path": (
                f"mobile_package/{path.name}"
                if path.parent == PACKAGE_ROOT else path.name
            ),
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        }
        for path in manifest_files
    ],
    "test_set_status": "sealed",
    "test_set_used": False,
})

print("Status                 :", lock["status"].upper())
print("NI boundary clearance  :", boundary_clearance)
print("Strict verbatim gate   : FAILED / PRESERVED")
print("Production release     : NOT AUTHORIZED")
print("Mobile benchmark       : AUTHORIZED")
print("Held-out Test          : SEALED / NOT USED")
print("Parallel next          :", lock["parallel_next_phases"])
print("Output                 :", WORK_ROOT)

Status                 : LOCKED_FOR_CONDITIONAL_MOBILE_BENCHMARK
NI boundary clearance  : 4.8737216705226766e-06
Strict verbatim gate   : FAILED / PRESERVED
Production release     : NOT AUTHORIZED
Mobile benchmark       : AUTHORIZED
Held-out Test          : SEALED / NOT USED
Parallel next          : ['F30K_T3S3_Android_Benchmark', 'F30K_T3S3_INT8_Qualitative']
Output                 : /kaggle/working/F30K_T3S3_Deployment_Lock
